# dataclass-training-args — faded example 2: Repair a mutable list default with default_factory

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataclass-training-args`. Running the beacon reports progress on the `Config: @dataclass training args` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: @dataclass training args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclass-training-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclass-training-args"
DD_SUBTOPIC = "Config: @dataclass training args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A bare mutable default like `tags: list = []` is shared across all instances and raises `ValueError` at class-definition time in a dataclass. The fix is `field(default_factory=...)`, which calls the factory fresh for each new instance so every object gets its own independent list.

## Faded exercise 2

### Faded — independent list default via `default_factory`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> ```

Complete the `LoggerArgs` dataclass so its `tags` field defaults to `['baseline']` but gives each instance its OWN list (so appending to one instance's `tags` never leaks into another). Use `field(default_factory=...)`. The scalar fields and the import are already provided.

**Fill in:** the `tags` field declaration using field(default_factory=...) so each instance gets a fresh ['baseline'] list.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class LoggerArgs:
    project: str = 'demo'
    log_every: int = 50
    tags = None  # TODO: tags: list defaulting to ['baseline'] via field(default_factory=...)

a = LoggerArgs()
a.tags.append('exp-a')
b = LoggerArgs()
print(a.tags)
print(b.tags)


def _test():
    a = LoggerArgs()
    b = LoggerArgs()
    assert a.tags == ['baseline'], a.tags
    assert b.tags == ['baseline'], b.tags
    assert a.tags is not b.tags, 'instances must not share the same list object'
    a.tags.append('exp-a')
    assert a.tags == ['baseline', 'exp-a'], a.tags
    assert b.tags == ['baseline'], 'mutation leaked across instances'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass, field

@dataclass
class LoggerArgs:
    project: str = 'demo'
    log_every: int = 50
    tags: list = field(default_factory=lambda: ['baseline'])

a = LoggerArgs()
a.tags.append('exp-a')
b = LoggerArgs()
print(a.tags)
print(b.tags)
```
</details>